# 단풍도서관 RAG · 문장 임베딩으로 신규 청크의 의미적 이상치 찾기

## 1단계 · 현행 문서로 답하기

가상의 단풍도서관에서 **일반 도서의 기본 대출 기간은 며칠인가요?** 현행 안내문 열 개를 문단 단위로 나누고, FAISS가 찾은 상위 청크 세 개를 한 번에 생성 모델에 전달한다. 한국어 질문, 검색 결과, 프롬프트, 답변을 함께 기록해 이후 실험의 기준으로 쓴다.

Python 3 환경에서 필요한 경우 아래 설치 명령의 주석을 풀고, 위에서부터 모든 셀을 실행한다. 처음 실행할 때 답변 생성 모델과 문장 임베딩 모델을 내려받는다. 캐시에 모델이 있으면 CPU에서도 오프라인으로 실행할 수 있다.

In [1]:
# %pip install faiss-cpu scikit-learn numpy torch "transformers>=4.51"

## 문서와 청크

각 문서는 다른 상황에서 독자가 접하는 안내문이다. 빈 줄로 나뉜 문단 하나를 청크 하나로 사용한다. 청크 ID 앞부분은 원문 문서 ID이고 뒷부분은 그 안의 문단 번호다.

In [2]:
documents = {
    "이용규정": """일반 도서의 기본 대출 기간은 14일입니다.

참고 도서는 도서관 안에서만 열람할 수 있습니다.""",
    "회원안내": """회원은 일반 도서를 14일 동안 빌릴 수 있습니다.

도서관 회원증을 신청하려면 안내 데스크에 사진이 있는 신분증을 제시하세요.""",
    "대출영수증": """일반 도서의 대출 기한은 대출일로부터 14일입니다.

대출 영수증에는 빌린 책의 제목과 반납 예정일이 표시됩니다.""",
    "반납안내": """야외 도서 반납함은 도서관 정문 바로 옆에 있습니다.

대형 자료는 안내 데스크에 직접 반납해 주세요.""",
    "운영시간": """단풍도서관의 주말 개관 시간은 오전 10시입니다.

평일에는 오전 9시에 문을 열고 오후 8시에 닫습니다.""",
    "예약안내": """다른 회원이 빌린 책은 도서관 누리집에서 예약할 수 있습니다.

예약한 책이 도착하면 문자로 안내합니다.""",
    "연장안내": """예약 대기자가 없는 책은 한 번 연장할 수 있습니다.

연장은 반납 예정일 전에 누리집의 내 서재에서 신청하세요.""",
    "열람실안내": """조용한 열람실은 본관 2층에 있습니다.

열람실 좌석은 입구의 단말기에서 선택하세요.""",
    "복사안내": """흑백 복사기는 본관 1층 자료실에 있습니다.

흑백 복사 요금은 한 면에 50원입니다.""",
    "분실물안내": """도서관에서 잃어버린 물건은 안내 데스크에 문의하세요.

접수한 분실물은 한 달 동안 보관합니다.""",
}

def split_documents(documents):
    return [
        {"id": f"{doc_id}/{number}", "doc_id": doc_id, "text": paragraph.strip()}
        for doc_id, text in documents.items()
        for number, paragraph in enumerate(text.split("\n\n"), start=1)
        if paragraph.strip()
    ]

chunks = split_documents(documents)
print(f"문서 {len(documents)}개 → 청크 {len(chunks)}개")
for chunk in chunks:
    print(f"{chunk['id']} · {chunk['text']}")

문서 10개 → 청크 20개
이용규정/1 · 일반 도서의 기본 대출 기간은 14일입니다.
이용규정/2 · 참고 도서는 도서관 안에서만 열람할 수 있습니다.
회원안내/1 · 회원은 일반 도서를 14일 동안 빌릴 수 있습니다.
회원안내/2 · 도서관 회원증을 신청하려면 안내 데스크에 사진이 있는 신분증을 제시하세요.
대출영수증/1 · 일반 도서의 대출 기한은 대출일로부터 14일입니다.
대출영수증/2 · 대출 영수증에는 빌린 책의 제목과 반납 예정일이 표시됩니다.
반납안내/1 · 야외 도서 반납함은 도서관 정문 바로 옆에 있습니다.
반납안내/2 · 대형 자료는 안내 데스크에 직접 반납해 주세요.
운영시간/1 · 단풍도서관의 주말 개관 시간은 오전 10시입니다.
운영시간/2 · 평일에는 오전 9시에 문을 열고 오후 8시에 닫습니다.
예약안내/1 · 다른 회원이 빌린 책은 도서관 누리집에서 예약할 수 있습니다.
예약안내/2 · 예약한 책이 도착하면 문자로 안내합니다.
연장안내/1 · 예약 대기자가 없는 책은 한 번 연장할 수 있습니다.
연장안내/2 · 연장은 반납 예정일 전에 누리집의 내 서재에서 신청하세요.
열람실안내/1 · 조용한 열람실은 본관 2층에 있습니다.
열람실안내/2 · 열람실 좌석은 입구의 단말기에서 선택하세요.
복사안내/1 · 흑백 복사기는 본관 1층 자료실에 있습니다.
복사안내/2 · 흑백 복사 요금은 한 면에 50원입니다.
분실물안내/1 · 도서관에서 잃어버린 물건은 안내 데스크에 문의하세요.
분실물안내/2 · 접수한 분실물은 한 달 동안 보관합니다.


## FAISS 검색

조사가 붙어도 ‘반납함’과 ‘반납함은’의 공통 부분을 찾을 수 있도록 어절 안에서 3~5글자씩 겹쳐 읽는다. 어절 양끝의 공백도 글자 수에 포함한다. 질문과 청크를 이 문자 조각의 TF-IDF 벡터로 표현하고 L2 정규화한다. FAISS가 계산한 내적은 코사인 유사도이며, 점수가 0보다 큰 청크를 최대 $k$개 반환한다.

In [3]:
import faiss
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

class PassageIndex:
    def __init__(self, chunks, *, vectorizer=None):
        chunks = list(chunks)
        if vectorizer is None:
            vectorizer = TfidfVectorizer(
                analyzer="char_wb", ngram_range=(3, 5), dtype=np.float32, norm="l2"
            )
            vectorizer.fit([chunk["text"] for chunk in chunks])
        self.vectorizer = vectorizer
        self.index = faiss.IndexFlatIP(len(self.vectorizer.vocabulary_))
        self.chunks = []
        self.add_chunks(chunks)

    def add_chunks(self, chunks):
        batch = list(chunks)
        if not batch:
            return
        new_ids = {chunk["id"] for chunk in batch}
        if len(new_ids) != len(batch) or any(chunk["id"] in new_ids for chunk in self.chunks):
            raise ValueError("같은 청크 ID가 두 번 등록됩니다.")
        vectors = self.vectorizer.transform([chunk["text"] for chunk in batch])
        self.index.add(vectors.toarray())
        self.chunks.extend(batch)

    def search(self, question, k=3):
        if k <= 0 or self.index.ntotal == 0:
            return []
        query = self.vectorizer.transform([question])
        if query.nnz == 0:
            return []
        scores, positions = self.index.search(query.toarray(), min(k, self.index.ntotal))
        return [
            {**self.chunks[position], "score": float(score)}
            for score, position in zip(scores[0], positions[0])
            if score > 0
        ]

retriever = PassageIndex(chunks)
question = "일반 도서의 기본 대출 기간은 며칠인가요?"
hits = retriever.search(question, k=3)

print(question)
for rank, hit in enumerate(hits, start=1):
    print(f"{rank}. [{hit['score']:.3f}] {hit['id']} · {hit['text']}")

일반 도서의 기본 대출 기간은 며칠인가요?
1. [0.779] 이용규정/1 · 일반 도서의 기본 대출 기간은 14일입니다.
2. [0.329] 대출영수증/1 · 일반 도서의 대출 기한은 대출일로부터 14일입니다.
3. [0.103] 회원안내/1 · 회원은 일반 도서를 14일 동안 빌릴 수 있습니다.


## 검색한 근거로 답변 생성

상위 청크들을 하나의 한국어 프롬프트에 담아 Qwen3-0.6B를 한 번 호출한다. 같은 조건에서 결과를 비교할 수 있도록 모델 버전을 고정하고, 비사고 모드에서 매번 확률이 가장 높은 토큰을 선택한다. 검색 결과가 비면 모델 호출을 건너뛴다. [Qwen3-0.6B 모델 카드](https://huggingface.co/Qwen/Qwen3-0.6B)

In [4]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.utils import logging as model_logging

model_logging.disable_progress_bar()
torch.set_num_threads(min(torch.get_num_threads(), 4))
model_id = "Qwen/Qwen3-0.6B"
revision = "c1899de289a04d12100db370d81485cdf75e47ca"
tokenizer = AutoTokenizer.from_pretrained(model_id, revision=revision)
model = AutoModelForCausalLM.from_pretrained(
    model_id, revision=revision, use_safetensors=True
).eval()

def make_prompt(question, passages):
    context = "\n".join(passage["text"] for passage in passages)
    return (
        f"다음은 도서관 안내문입니다.\n\n{context}\n\n질문: {question}\n"
        "안내문에서 질문의 답을 찾아 한국어로 한 문장으로 답하세요."
    )

@torch.inference_mode()
def answer_from(question, passages):
    if not passages:
        return "근거 없음"
    messages = [{"role": "user", "content": make_prompt(question, passages)}]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, enable_thinking=False,
        return_tensors="pt", return_dict=True,
    )
    output = model.generate(**inputs, do_sample=False, max_new_tokens=48)
    answer_tokens = output[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(answer_tokens, skip_special_tokens=True).strip()

print(make_prompt(question, hits))
answer = answer_from(question, hits)
print(f"\n모델 답변 → {answer}")

다음은 도서관 안내문입니다.

일반 도서의 기본 대출 기간은 14일입니다.
일반 도서의 대출 기한은 대출일로부터 14일입니다.
회원은 일반 도서를 14일 동안 빌릴 수 있습니다.

질문: 일반 도서의 기본 대출 기간은 며칠인가요?
안내문에서 질문의 답을 찾아 한국어로 한 문장으로 답하세요.



모델 답변 → 일반 도서의 기본 대출 기간은 14일입니다.


## 기준 결과 확인

대출 기간을 뒷받침하는 청크 세 개가 모두 검색됐는지, 생성 답변에 올바른 기간과 반납함 위치가 담겼는지 검사한다. 문서에 없는 망원경 질문은 검색 결과가 비어 ‘근거 없음’으로 처리되는지도 확인한다. 추가한 운영시간 안내문으로 주말 개관 시간도 확인한다.

In [5]:
assert len(documents) == 10
assert retriever.index.ntotal == len(chunks) == 20
assert hits[0]["id"] == "이용규정/1"
assert {hit["id"] for hit in hits} == {"이용규정/1", "회원안내/1", "대출영수증/1"}
assert "14일" in answer.replace(" ", "") or "열나흘" in answer, answer

location_question = "야외 도서 반납함은 어디에 있나요?"
location_hits = retriever.search(location_question)
location_answer = answer_from(location_question, location_hits)
print(f"{location_question} → {location_answer} [{location_hits[0]['id']}]")
assert location_hits[0]["id"] == "반납안내/1"
assert "정문" in location_answer and "옆" in location_answer, location_answer

unknown_question = "망원경이 있나요?"
unknown_hits = retriever.search(unknown_question)
unknown_answer = answer_from(unknown_question, unknown_hits)
print(f"{unknown_question} → {unknown_answer} (검색 {len(unknown_hits)}건)")
assert not unknown_hits and unknown_answer == "근거 없음"

hours_question = "단풍도서관의 주말 개관 시간은 언제인가요?"
hours_hits = retriever.search(hours_question)
hours_answer = answer_from(hours_question, hours_hits)
print(f"{hours_question} → {hours_answer} [{hours_hits[0]['id']}]")
assert hours_hits[0]["id"] == "운영시간/1"
assert "오전10시" in hours_answer.replace(" ", ""), hours_answer

야외 도서 반납함은 어디에 있나요? → 야외 도서 반납함은 도서관 정문 바로 옆에 있습니다. [반납안내/1]
망원경이 있나요? → 근거 없음 (검색 0건)


단풍도서관의 주말 개관 시간은 언제인가요? → 단풍도서관의 주말 개관 시간은 오전 10시입니다. [운영시간/1]


## 2단계 · 신규 문서에 허위 규정이 섞인 상황

기존 도서관 안내문 10건(20개 청크)은 검수된 참조 자료라고 가정한다. 신규 문서 네 건 중 하나는 이용자에게 잘못된 대출 기간을 알려주려는 가상의 허위 안내다. 같은 배치에 도서관 업무와 무관한 문장 하나와 정상 안내 두 건을 포함한다. 정상 안내에는 기존 자료에서 드물었던 접근성 서비스도 넣어, 새 주제를 잘못 제외하는지 살펴본다.

아래 문서 ID는 추적용이다. 필터는 본문의 문장 임베딩만 사용한다.

In [6]:
incoming_documents = {
    "접수-A": "일반 도서의 기본 대출 기간은 30일입니다.",
    "접수-B": "토성의 고리는 주로 얼음과 암석 입자로 이루어져 있습니다.",
    "접수-C": "반납한 책은 안내 데스크에서 처리한 뒤 서가에 다시 꽂습니다.",
    "접수-D": "청각장애인용 진동 호출기는 1층 창구에서 빌릴 수 있습니다.",
}
incoming_chunks = split_documents(incoming_documents)
mixed_retriever = PassageIndex(chunks + incoming_chunks, vectorizer=retriever.vectorizer)
print(f"기존 {len(chunks)}개 + 신규 {len(incoming_chunks)}개 = {mixed_retriever.index.ntotal}개 청크")
for chunk in incoming_chunks:
    print(f"{chunk['id']} · {chunk['text']}")

기존 20개 + 신규 4개 = 24개 청크
접수-A/1 · 일반 도서의 기본 대출 기간은 30일입니다.
접수-B/1 · 토성의 고리는 주로 얼음과 암석 입자로 이루어져 있습니다.
접수-C/1 · 반납한 책은 안내 데스크에서 처리한 뒤 서가에 다시 꽂습니다.
접수-D/1 · 청각장애인용 진동 호출기는 1층 창구에서 빌릴 수 있습니다.


## 3단계 · 임베딩 거리의 상위 꼬리 잘라내기

여러 주제가 섞인 도서관 자료에서 신규 청크가 기존 자료의 어느 주제와 가까운지 본다. 한국어를 지원하는 multilingual-e5-small로 본문을 임베딩하고 길이를 1로 정규화한다. 모델 카드의 문장 간 유사도 지침에 따라 비교하는 양쪽에 동일한 `query: ` 접두사를 붙인다. FAISS 검색과 답변 생성 조건은 1단계와 같다. [임베딩 모델 카드](https://huggingface.co/intfloat/multilingual-e5-small)

**이상 점수**는 가장 가까운 참조 청크 3개까지의 평균 코사인 거리다. 점수가 클수록 주변에 비슷한 근거가 드물다. 기준을 정할 때는 참조 청크마다 자기 문서의 모든 청크를 이웃에서 제외한다. 이렇게 얻은 참조 점수의 95백분위수를 넘는 신규 청크만 격리한다. 이웃 수와 백분위수는 이 소규모 실험에서 미리 정한 값이다.

In [7]:
from transformers import AutoModel

encoder_id = "intfloat/multilingual-e5-small"
encoder_revision = "614241f622f53c4eeff9890bdc4f31cfecc418b3"
embed_tokenizer = AutoTokenizer.from_pretrained(encoder_id, revision=encoder_revision)
encoder = AutoModel.from_pretrained(
    encoder_id, revision=encoder_revision, use_safetensors=True
).eval()

@torch.inference_mode()
def embed_chunks(passages):
    texts = ["query: " + passage["text"] for passage in passages]
    inputs = embed_tokenizer(texts, padding=True, truncation=True, max_length=512, return_tensors="pt")
    hidden = encoder(**inputs).last_hidden_state
    mask = inputs["attention_mask"].unsqueeze(-1)
    pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1)
    return torch.nn.functional.normalize(pooled, dim=1).cpu().numpy()

def cosine_distances(vectors, references):
    return 1 - np.clip(vectors @ references.T, -1.0, 1.0)

def mean_neighbor_distance(distances, k):
    if k < 1 or np.any(np.isfinite(distances).sum(axis=1) < k):
        raise ValueError("점수를 계산하려면 각 행에 참조 이웃이 k개 이상 있어야 합니다.")
    return np.sort(distances, axis=1)[:, :k].mean(axis=1)

k_neighbors = 3
quantile = 0.95
reference_vectors = embed_chunks(chunks)
reference_distances = cosine_distances(reference_vectors, reference_vectors)
doc_ids = np.array([chunk["doc_id"] for chunk in chunks])
reference_distances[doc_ids[:, None] == doc_ids[None, :]] = np.inf
reference_scores = mean_neighbor_distance(reference_distances, k_neighbors)
cutoff = np.quantile(reference_scores, quantile)
print(f"참조 청크 {len(chunks)}개 · 이웃 {k_neighbors}개 · 95백분위 거리 {cutoff:.4f}")

참조 청크 20개 · 이웃 3개 · 95백분위 거리 0.1531


In [8]:
distances = cosine_distances(embed_chunks(incoming_chunks), reference_vectors)
scores = mean_neighbor_distance(distances, k_neighbors)
trimmed = scores > cutoff
accepted_chunks = [chunk for chunk, flag in zip(incoming_chunks, trimmed) if not flag]
trimmed_retriever = PassageIndex(chunks + accepted_chunks, vectorizer=retriever.vectorizer)

for chunk, row, score, flag in zip(incoming_chunks, distances, scores, trimmed):
    nearest = chunks[row.argmin()]
    decision = "격리" if flag else "통과"
    print(f"{chunk['id']} · {decision} · 거리 {score:.4f} · 가장 가까운 청크 {nearest['id']}")
    print(f"  신규: {chunk['text']}\n  참조: {nearest['text']}")
print(f"검색 대상 청크 {mixed_retriever.index.ntotal}개 → {trimmed_retriever.index.ntotal}개")
assert trimmed_retriever.index.ntotal == len(chunks) + len(accepted_chunks)

접수-A/1 · 통과 · 거리 0.0599 · 가장 가까운 청크 이용규정/1
  신규: 일반 도서의 기본 대출 기간은 30일입니다.
  참조: 일반 도서의 기본 대출 기간은 14일입니다.
접수-B/1 · 격리 · 거리 0.1907 · 가장 가까운 청크 열람실안내/2
  신규: 토성의 고리는 주로 얼음과 암석 입자로 이루어져 있습니다.
  참조: 열람실 좌석은 입구의 단말기에서 선택하세요.
접수-C/1 · 통과 · 거리 0.0881 · 가장 가까운 청크 반납안내/2
  신규: 반납한 책은 안내 데스크에서 처리한 뒤 서가에 다시 꽂습니다.
  참조: 대형 자료는 안내 데스크에 직접 반납해 주세요.
접수-D/1 · 통과 · 거리 0.1438 · 가장 가까운 청크 열람실안내/1
  신규: 청각장애인용 진동 호출기는 1층 창구에서 빌릴 수 있습니다.
  참조: 조용한 열람실은 본관 2층에 있습니다.
검색 대상 청크 24개 → 23개


### 트림을 통과한 수치 충돌 확인

주제가 도서 대출이라는 점에서는 ‘14일’과 ‘30일’ 문장이 서로 가깝다. 트림 단독 결과를 보존하고, 이 사례를 위한 보조 검사를 별도 조건으로 비교한다. 신규 청크와 의미상 가까운 참조 청크 3개에서 정수를 제외한 글자와 단위가 모두 일치하는 문장을 찾는다. 같은 문장에 서로 다른 수치가 있으면 근거 ID를 남기고 검색에서 보류한다.

이 검사의 적용 범위는 문장 틀이 같은 수치 규정이다. 바꿔 쓴 문장이나 새로운 규정의 진위는 이 검사로 판별하기 어렵다.

In [9]:
import re

def has_numeric_conflict(text, reference):
    parts = re.split(r"([0-9]+)", text.strip())
    other = re.split(r"([0-9]+)", reference.strip())
    if len(parts) == 1 or parts[::2] != other[::2]:
        return False
    return tuple(map(int, parts[1::2])) != tuple(map(int, other[1::2]))

number_conflicts = []
for chunk, row in zip(incoming_chunks, distances):
    peers = [chunks[i] for i in np.argsort(row)[:k_neighbors]]
    conflicts = [p for p in peers if has_numeric_conflict(chunk["text"], p["text"])]
    number_conflicts.append(bool(conflicts))
    for peer in conflicts:
        print(f"{chunk['id']} · {chunk['text']}")
        print(f"  수치 대조 근거 · {peer['id']} · {peer['text']}")

combined = trimmed | np.array(number_conflicts)
retained = [c for c, flag in zip(incoming_chunks, combined) if not flag]
cleaned_retriever = PassageIndex(chunks + retained, vectorizer=retriever.vectorizer)
print(f"트림만 적용한 인덱스 {trimmed_retriever.index.ntotal}개 · 수치 대조까지 적용한 인덱스 {cleaned_retriever.index.ntotal}개")

접수-A/1 · 일반 도서의 기본 대출 기간은 30일입니다.
  수치 대조 근거 · 이용규정/1 · 일반 도서의 기본 대출 기간은 14일입니다.
트림만 적용한 인덱스 23개 · 수치 대조까지 적용한 인덱스 22개


### 탐지 결과와 검색 답변 확인

아래 정답표는 점수 계산이 끝난 뒤에만 사용한다. 허위 규정과 무관 자료는 각각 구분해 기록하고, 정상 청크를 잘못 격리한 수도 센다. 검색 비교의 기준 인덱스에는 검수 문서와 신규 정상 문서를 모두 넣는다. 네 조건에서 같은 질문으로 상위 3개 근거와 모델 답변을 확인한다.

In [10]:
truth = {
    "접수-A": "허위 규정",
    "접수-B": "무관 자료",
    "접수-C": "정상",
    "접수-D": "정상",
}
labels = np.array([truth[chunk["doc_id"]] for chunk in incoming_chunks])
for label in ("허위 규정", "무관 자료", "정상"):
    in_group = labels == label
    print(f"{label} {in_group.sum()}개 · 트림만 격리 {np.sum(trimmed & in_group)}개"
          f" · 수치 대조 포함 격리 {np.sum(combined & in_group)}개")

reviewed_chunks = [c for c, label in zip(incoming_chunks, labels) if label == "정상"]
reviewed_retriever = PassageIndex(chunks + reviewed_chunks, vectorizer=retriever.vectorizer)
reference_ids = {chunk["id"] for chunk in chunks}
for index, flags in [(trimmed_retriever, trimmed), (cleaned_retriever, combined)]:
    kept_ids = {c["id"] for c, flag in zip(incoming_chunks, flags) if not flag}
    assert {c["id"] for c in index.chunks} == reference_ids | kept_ids

conditions = [
    ("정상만", reviewed_retriever),
    ("혼합", mixed_retriever),
    ("트림만", trimmed_retriever),
    ("트림과 수치 대조", cleaned_retriever),
]
for query in (question, hours_question):
    print(f"\n질문 · {query}")
    for name, index in conditions:
        passages = index.search(query, k=3)
        ranking = " → ".join(f"{p['id']} ({p['score']:.3f})" for p in passages)
        print(f"  {name} · {ranking}")
        print(f"    {answer_from(query, passages)}")

허위 규정 1개 · 트림만 격리 0개 · 수치 대조 포함 격리 1개
무관 자료 1개 · 트림만 격리 1개 · 수치 대조 포함 격리 1개
정상 2개 · 트림만 격리 0개 · 수치 대조 포함 격리 0개

질문 · 일반 도서의 기본 대출 기간은 며칠인가요?
  정상만 · 이용규정/1 (0.779) → 대출영수증/1 (0.329) → 회원안내/1 (0.103)


    일반 도서의 기본 대출 기간은 14일입니다.
  혼합 · 접수-A/1 (0.891) → 이용규정/1 (0.779) → 대출영수증/1 (0.329)


    일반 도서의 기본 대출 기간은 14일입니다.
  트림만 · 접수-A/1 (0.891) → 이용규정/1 (0.779) → 대출영수증/1 (0.329)


    일반 도서의 기본 대출 기간은 14일입니다.
  트림과 수치 대조 · 이용규정/1 (0.779) → 대출영수증/1 (0.329) → 회원안내/1 (0.103)


    일반 도서의 기본 대출 기간은 14일입니다.

질문 · 단풍도서관의 주말 개관 시간은 언제인가요?
  정상만 · 운영시간/1 (0.785) → 이용규정/1 (0.029) → 반납안내/1 (0.014)


    단풍도서관의 주말 개관 시간은 오전 10시입니다.
  혼합 · 운영시간/1 (0.785) → 접수-A/1 (0.033) → 이용규정/1 (0.029)


    단풍도서관의 주말 개관 시간은 오전 10시입니다.
  트림만 · 운영시간/1 (0.785) → 접수-A/1 (0.033) → 이용규정/1 (0.029)


    단풍도서관의 주말 개관 시간은 오전 10시입니다.
  트림과 수치 대조 · 운영시간/1 (0.785) → 이용규정/1 (0.029) → 반납안내/1 (0.014)


    단풍도서관의 주말 개관 시간은 오전 10시입니다.


### 관찰 결과

신규 네 건 중 토성에 관한 문장만 임베딩 트림 기준(거리 0.1531)을 넘었다. 대출 기간을 30일로 적은 허위 규정은 거리 0.0599로 통과했고, 대출 질문의 검색 결과에서도 1위에 남았다. 수치 대조를 추가하자 검수된 14일 규정과의 충돌이 드러나 해당 청크를 보류했다. 정상 안내 두 건은 두 조건 모두 통과했다.

모델은 네 조건에서 모두 대출 기간을 14일이라고 답했다. 이번에 바뀐 것은 답변의 근거였다. 네 건은 주제 이탈과 수치 충돌의 차이를 관찰하기 위한 예시다. 새로운 주제의 정상 문서에 대한 오탐률을 추정하려면 별도의 평가 자료가 필요하다.